In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

import os
cwd = os.getcwd()

from geopy.geocoders import Nominatim
import zipfile
from geopy.extra.rate_limiter import RateLimiter
import geopandas as gpd
import plotly.express as px
from shapely.geometry import Point

from IPython.display import Image
from IPython.display import IFrame

import nbconvert
from nbconvert import HTMLExporter
import nbformat

In [2]:
def state_countyMetrics(state_name, fips_code):

    #get my data from the zip and from the pickle data 
    df_cancer_rates = pd.read_csv(os.path.join(cwd, 'data', 'county_zip_reg.csv'), encoding='latin1')
    df_nitrogen_samples = pd.read_pickle('data/us_locations.pkl')

    #get my polution data
    state_nitrogen = nitrogen_samples(df_nitrogen_samples, fips_code, state_name)

    #filter for only those whose county field (County, State) contains state name
    filtered_df = df_cancer_rates[df_cancer_rates['County'].str.contains(state_name, case=False, na=False)]

    countyTotals = filtered_df.groupby('County').agg({
        'incidenceRate': 'mean',
        'deathRate': 'mean',
        'avgDeathsPerYear': 'mean',
    }).reset_index()

    #drop the parish/county designation (Acadiana Parish -> Acadiana, Harris County -> Harris) to match other formats
    countyTotals['County'] = countyTotals['County'].str.split('Parish').str[0].str.strip()
    countyTotals['County'] = countyTotals['County'].str.split('County').str[0].str.strip()

    #prevent capitilization issues by forcing to capital
    countyTotals['County'] = countyTotals['County'].str.title()

    if colab == True:
      zip_path = os.path.join(cwd, 'cb_2018_us_county_500k.zip')

      with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("county_shapefiles")

      counties = gpd.read_file(os.path.join(cwd, 'county_shapefiles','cb_2018_us_county_500k.shp'), encoding='latin1')

    else:
      with zipfile.ZipFile(os.path.join(cwd, "data/cb_2018_us_county_500k.zip"), "r") as zip_ref:
        zip_ref.extractall("county_shapefiles")

      counties = gpd.read_file(os.path.join(cwd, 'county_shapefiles','cb_2018_us_county_500k.shp'), encoding='latin1')

    #fips code for counties data from json
    la_counties = counties[counties['STATEFP'] == fips_code]

    #merge the nitrogen samples dataset and cancer rates dataset on the county column
    merged = la_counties.merge(countyTotals, left_on='NAME', right_on='County')

    merged['Nitrogen (mg/L)'] = state_nitrogen['Nitrogen (mg/L)']
    merged['Phosphorus (mg/L)'] = state_nitrogen['Phosphorus (mg/L)']
    
    return merged

def nitrogen_plotter(merged):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    merged.plot(column='Nitrogen (mg/L)', cmap='vlag', linewidth=0.8, ax=ax1,
            edgecolor='0.8', legend=True,
            missing_kwds={
                "color": "lightgrey",
                "edgecolor": "black",
                "hatch": "///",
                "label": "No data"
            })
    #points_gdf.plot(ax=ax1, color='purple', markersize=150)

    ax1.set_title("Nitrogen by County (m/g L)", fontsize=16)

    ax1.set_axis_off()

    merged.plot(column='Phosphorus (mg/L)', cmap='vlag', linewidth=0.8, ax=ax2,
            edgecolor='0.8', legend=True,
            missing_kwds={
                "color": "lightgrey",
                "edgecolor": "black",
                "hatch": "///",
                "label": "No data"
            })

    #points_gdf.plot(ax=ax2, color='black', markersize=50)

    ax2.set_title('Phosphorus by County')
    ax2.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
df_us = pd.read_pickle('data/us_locations.pkl')

#print(tabulate(df_us, headers='keys', tablefmt='github', showindex=False))

us_states = gpd.read_file('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json')
merged_measures = us_states.merge(df_us, left_on='name', right_on='State')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7))

merged_measures.plot(
    column='Phosphorus (mg/L)', cmap='vlag', linewidth=0.8,
    ax=ax1, edgecolor='0.8', legend=True
)

ax1.set_title('Phosphorus (mg/L)')
ax1.axis('off')

merged_measures.plot(
    column='Nitrogen (mg/L)', cmap='vlag', linewidth=0.8,
    ax=ax2, edgecolor='0.8', legend=True
)

ax2.set_title('Nitrogen (mg/L)')
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
df = pd.read_csv(os.path.join(cwd, 'data', 'cancer_reg.csv'))

#drop the city designation and just keep the state
df['State'] = df['geography'].str.split(',').str[-1].str.strip()

#group by state and average - will use this later
state_avg = df.groupby('State').mean(numeric_only=True).reset_index()


#load a graphic of the US using geoJSON
us_states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")

#merge in the state averages to the map
merged = us_states.merge(state_avg, left_on='name', right_on='State')

#set up 2x2 grid for future scalability (only using 2 now)
fig, axes = plt.subplots(2, 2, figsize=(10, 5))
axes = axes.flatten()

merged_measures.plot(
    column='Phosphorus (mg/L)', cmap='vlag', linewidth=0.8,
    ax=axes[0], edgecolor='0.8', legend=True
)
axes[0].set_title('Phosphorus (mg/L)')
axes[0].axis('off')

merged_measures.plot(
    column='Nitrogen (mg/L)', cmap='vlag', linewidth=0.8,
    ax=axes[1], edgecolor='0.8', legend=True
)
axes[1].set_title('Nitrogen (mg/L)')
axes[1].axis('off')

merged.plot(column='avgdeathsperyear', cmap='vlag', linewidth=0.8,
            ax=axes[2], edgecolor='0.8', legend=True)
axes[2].set_title('Deaths per year')
axes[2].axis('off')

merged.plot(column='incidencerate', cmap='vlag', linewidth=0.8,
            ax=axes[3], edgecolor='0.8', legend=True)
axes[3].set_title('Incidence Rate')
axes[3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def get_county(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en', exactly_one=True, timeout=10)
        if location and 'address' in location.raw:
            address = location.raw['address']
            return address.get('county') or address.get('parish') or address.get('state_district') or address.get('region')

        return None

    except Exception as e:
        print(f"Error at ({lat},{lon}): {e}")
        return None



def nitrogen_samples(df, fips_code, state_name):
    
    #get only data for our desired state
    df_state_nitrogen = df[df['State'].str.contains(state_name, case=False, na=False)].copy()

    #acquire county from the lat lon coordinates - use fancy lambda for list iteration and string indexing
    df_state_nitrogen.loc[:, 'County'] = df_state_nitrogen.apply(
    lambda row: get_county(row['Geographical Location (Latitude)'], row['Geographical Location (Longitude)']),
    axis=1)

    #group all county entries together to get totals by county
    countyTotals = df_state_nitrogen.groupby('County').agg({
    'Nitrogen (mg/L)': 'mean',
    'Phosphorus (mg/L)': 'mean'
    }).reset_index()

    #could do some kind of evaluation here - but rather explicitly search for both in case it uses a parish system
    countyTotals['County'] = countyTotals['County'].str.split('Parish').str[0].str.strip()
    countyTotals['County'] = countyTotals['County'].str.split('County').str[0].str.strip()

    #avoid capitilization issues between strings 
    countyTotals['County'] = countyTotals['County'].str.title()

    #print(countyTotals)

    if colab == True:
      zip_path = os.path.join(cwd, 'cb_2018_us_county_500k.zip')

      with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("county_shapefiles")

      counties = gpd.read_file(os.path.join(cwd, 'county_shapefiles','cb_2018_us_county_500k.shp'), encoding='latin1')

    else:
      with zipfile.ZipFile(os.path.join(cwd, "data/cb_2018_us_county_500k.zip"), "r") as zip_ref:
          zip_ref.extractall("county_shapefiles")

          counties = gpd.read_file(os.path.join(cwd, 'county_shapefiles','cb_2018_us_county_500k.shp'), encoding='latin1')

    #merge these together to plau nice with other data
    state_counties = counties[counties['STATEFP'] == fips_code]
    merged = state_counties.merge(countyTotals, left_on='NAME', right_on='County', how='left')

    return merged